<a href="https://colab.research.google.com/github/MohsinAshraf01/FA23-BAI-048-Computer-Vision/blob/main/skin_cancer_classification_with_model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch torchvision scikit-learn pandas numpy matplotlib seaborn xgboost torchinfo thop openpyxl

In [ ]:
!pip install thop

In [ ]:
# ============================================================
# SKIN CANCER CLASSIFICATION
# TABLE 1 + TABLE 2 + TABLE 3
# ============================================================

# ============================================================
# 1. INSTALL REQUIRED PACKAGES
# ============================================================

import sys
import subprocess
import importlib


def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name

    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )


install_if_missing("kagglehub")
install_if_missing("thop")
install_if_missing("xgboost")
install_if_missing("openpyxl")


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
import time
import copy
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

from thop import profile


warnings.filterwarnings("ignore")


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 4. SETTINGS
# ============================================================

DATASET_NAME = "nodoubttome/skin-cancer9-classesisic"

IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 0.0001

NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("DEVICE:", DEVICE)
print("=" * 70)


# ============================================================
# 5. DOWNLOAD DATASET AUTOMATICALLY
# ============================================================

print("\nDownloading / locating Kaggle dataset...")

DATASET_PATH = kagglehub.dataset_download(
    DATASET_NAME
)

print("\nDataset path:")
print(DATASET_PATH)


# ============================================================
# 6. FIND TRAIN AND TEST DIRECTORIES
# ============================================================

IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)


def contains_images(folder):
    try:
        for file in os.listdir(folder):
            if file.lower().endswith(IMAGE_EXTENSIONS):
                return True
    except:
        pass

    return False


def find_dataset_folders(base_path):

    train_candidates = []
    test_candidates = []

    for root, dirs, files in os.walk(base_path):

        folder_name = os.path.basename(root).lower()

        # ----------------------------------------------------
        # Look for folders named Train
        # ----------------------------------------------------
        if folder_name == "train":

            class_folders = []

            for d in dirs:

                class_path = os.path.join(root, d)

                if contains_images(class_path):
                    class_folders.append(class_path)

            if len(class_folders) >= 2:
                train_candidates.append(root)

        # ----------------------------------------------------
        # Look for folders named Test
        # ----------------------------------------------------
        if folder_name == "test":

            class_folders = []

            for d in dirs:

                class_path = os.path.join(root, d)

                if contains_images(class_path):
                    class_folders.append(class_path)

            if len(class_folders) >= 2:
                test_candidates.append(root)

    if len(train_candidates) == 0:
        raise FileNotFoundError(
            "Could not find the Train folder containing class folders."
        )

    train_root = train_candidates[0]

    test_root = None

    if len(test_candidates) > 0:
        test_root = test_candidates[0]

    return train_root, test_root


TRAIN_ROOT, TEST_ROOT = find_dataset_folders(
    DATASET_PATH
)


print("\n" + "=" * 70)
print("TRAIN ROOT:")
print(TRAIN_ROOT)

if TEST_ROOT:
    print("\nTEST ROOT:")
    print(TEST_ROOT)
else:
    print("\nNo separate Test folder found.")

print("=" * 70)


# ============================================================
# 7. TRANSFORMS
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# 8. LOAD DATASET
# ============================================================

train_full_dataset = datasets.ImageFolder(
    TRAIN_ROOT,
    transform=train_transform
)

test_dataset = None

if TEST_ROOT is not None:

    test_dataset = datasets.ImageFolder(
        TEST_ROOT,
        transform=eval_transform
    )


print("\nClasses found:")
print(train_full_dataset.classes)

NUM_CLASSES = len(train_full_dataset.classes)

print("\nNumber of classes:", NUM_CLASSES)
print("Training images:", len(train_full_dataset))

if test_dataset is not None:
    print("Test images:", len(test_dataset))


# ============================================================
# 9. CREATE TRAIN / VALIDATION SPLIT
# ============================================================

all_indices = np.arange(
    len(train_full_dataset)
)

all_targets = np.array(
    train_full_dataset.targets
)


train_indices, val_indices = train_test_split(
    all_indices,
    test_size=0.15,
    random_state=SEED,
    stratify=all_targets
)


# ============================================================
# 10. DATASET SUBSET WITH DIFFERENT TRANSFORMS
# ============================================================

class TransformSubset(Dataset):

    def __init__(
        self,
        base_dataset,
        indices,
        transform
    ):

        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):

        return len(self.indices)

    def __getitem__(self, idx):

        real_index = self.indices[idx]

        image_path, label = self.base_dataset.samples[
            real_index
        ]

        image = self.base_dataset.loader(
            image_path
        )

        if self.transform is not None:
            image = self.transform(image)

        return image, label


train_dataset = TransformSubset(
    train_full_dataset,
    train_indices,
    train_transform
)


val_dataset = TransformSubset(
    train_full_dataset,
    val_indices,
    eval_transform
)


# ============================================================
# 11. DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)


if test_dataset is not None:

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

else:

    test_loader = val_loader


print("\nTrain samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_loader.dataset))


# ============================================================
# 12. MODEL BUILDERS
# ============================================================

def build_model(model_name, num_classes):

    if model_name == "AlexNet":

        model = models.alexnet(
            weights=models.AlexNet_Weights.DEFAULT
        )

        model.classifier[6] = nn.Linear(
            model.classifier[6].in_features,
            num_classes
        )

    elif model_name == "VGG16":

        model = models.vgg16(
            weights=models.VGG16_Weights.DEFAULT
        )

        model.classifier[6] = nn.Linear(
            model.classifier[6].in_features,
            num_classes
        )

    elif model_name == "VGG19":

        model = models.vgg19(
            weights=models.VGG19_Weights.DEFAULT
        )

        model.classifier[6] = nn.Linear(
            model.classifier[6].in_features,
            num_classes
        )

    elif model_name == "ResNet18":

        model = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        )

        model.fc = nn.Linear(
            model.fc.in_features,
            num_classes
        )

    elif model_name == "ResNet50":

        model = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT
        )

        model.fc = nn.Linear(
            model.fc.in_features,
            num_classes
        )

    elif model_name == "ResNet101":

        model = models.resnet101(
            weights=models.ResNet101_Weights.DEFAULT
        )

        model.fc = nn.Linear(
            model.fc.in_features,
            num_classes
        )

    elif model_name == "DenseNet121":

        model = models.densenet121(
            weights=models.DenseNet121_Weights.DEFAULT
        )

        model.classifier = nn.Linear(
            model.classifier.in_features,
            num_classes
        )

    elif model_name == "EfficientNet-B0":

        model = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )

        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            num_classes
        )

    else:

        raise ValueError(
            f"Unknown model: {model_name}"
        )

    return model.to(DEVICE)


# ============================================================
# 13. TRAINING FUNCTION
# ============================================================

def train_model(model, train_loader, val_loader):

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    best_accuracy = 0.0

    best_state = copy.deepcopy(
        model.state_dict()
    )

    for epoch in range(EPOCHS):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            running_loss += (
                loss.item() * images.size(0)
            )

            _, predicted = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

        train_accuracy = (
            correct / total
        )

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        model.eval()

        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(
                    DEVICE,
                    non_blocking=True
                )

                labels = labels.to(
                    DEVICE,
                    non_blocking=True
                )

                outputs = model(images)

                _, predicted = torch.max(
                    outputs,
                    1
                )

                val_total += labels.size(0)

                val_correct += (
                    predicted == labels
                ).sum().item()

        val_accuracy = (
            val_correct / val_total
        )

        if val_accuracy > best_accuracy:

            best_accuracy = val_accuracy

            best_state = copy.deepcopy(
                model.state_dict()
            )

        print(
            f"Epoch [{epoch + 1}/{EPOCHS}] "
            f"Train Acc: {train_accuracy * 100:.2f}% "
            f"Val Acc: {val_accuracy * 100:.2f}%"
        )

    model.load_state_dict(
        best_state
    )

    return model


# ============================================================
# 14. EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model,
    loader,
    num_classes
):

    model.eval()

    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            outputs = model(images)

            probabilities = torch.softmax(
                outputs,
                dim=1
            )

            predictions = torch.argmax(
                probabilities,
                dim=1
            )

            all_labels.extend(
                labels.numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_probabilities.extend(
                probabilities.cpu().numpy()
            )

    all_labels = np.array(
        all_labels
    )

    all_predictions = np.array(
        all_predictions
    )

    all_probabilities = np.array(
        all_probabilities
    )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision = precision_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    try:

        auc = roc_auc_score(
            all_labels,
            all_probabilities,
            multi_class="ovr",
            average="weighted"
        )

    except:

        auc = np.nan

    return {
        "Accuracy (%)": accuracy * 100,
        "Precision (%)": precision * 100,
        "Recall (%)": recall * 100,
        "F1-Score (%)": f1 * 100,
        "AUC (%)": auc * 100
        if not np.isnan(auc)
        else np.nan
    }


# ============================================================
# 15. TABLE 1
# COMPARISON OF TRANSFER LEARNING MODELS
# ============================================================

MODEL_NAMES = [
    "AlexNet",
    "VGG16",
    "VGG19",
    "ResNet18",
    "ResNet50",
    "ResNet101",
    "DenseNet121",
    "EfficientNet-B0"
]


table1_results = []

trained_models = {}

best_model_name = None
best_model_accuracy = -1


print("\n")
print("=" * 70)
print("TABLE 1 - TRANSFER LEARNING MODELS")
print("=" * 70)


for model_name in MODEL_NAMES:

    print("\n" + "-" * 70)
    print("Training:", model_name)
    print("-" * 70)

    model = build_model(
        model_name,
        NUM_CLASSES
    )

    model = train_model(
        model,
        train_loader,
        val_loader
    )

    metrics = evaluate_model(
        model,
        test_loader,
        NUM_CLASSES
    )

    table1_results.append({
        "Model": model_name,
        "Accuracy (%)": metrics["Accuracy (%)"],
        "Precision (%)": metrics["Precision (%)"],
        "Recall (%)": metrics["Recall (%)"],
        "F1-Score (%)": metrics["F1-Score (%)"],
        "AUC (%)": metrics["AUC (%)"]
    })

    trained_models[model_name] = model

    if metrics["Accuracy (%)"] > best_model_accuracy:

        best_model_accuracy = metrics[
            "Accuracy (%)"
        ]

        best_model_name = model_name

    print(
        f"\n{model_name} Results:"
    )

    for key, value in metrics.items():

        print(
            f"{key}: {value:.2f}"
        )


table1_df = pd.DataFrame(
    table1_results
)


print("\n")
print("=" * 70)
print("TABLE 1 RESULTS")
print("=" * 70)

print(
    table1_df.to_string(
        index=False
    )
)


print(
    "\nBest model:",
    best_model_name
)

print(
    "Best accuracy:",
    f"{best_model_accuracy:.2f}%"
)


# ============================================================
# 16. FEATURE EXTRACTION
# ============================================================

best_model = trained_models[
    best_model_name
]


def get_feature_extractor(
    model,
    model_name
):

    if model_name == "AlexNet":

        feature_model = nn.Sequential(
            model.features,
            model.avgpool,
            nn.Flatten(),
            *list(model.classifier)[:-1]
        )

    elif model_name in [
        "VGG16",
        "VGG19"
    ]:

        feature_model = nn.Sequential(
            model.features,
            model.avgpool,
            nn.Flatten(),
            *list(model.classifier)[:-1]
        )

    elif model_name in [
        "ResNet18",
        "ResNet50",
        "ResNet101"
    ]:

        feature_model = nn.Sequential(
            model.conv1,
            model.bn1,
            model.relu,
            model.maxpool,
            model.layer1,
            model.layer2,
            model.layer3,
            model.layer4,
            model.avgpool,
            nn.Flatten()
        )

    elif model_name == "DenseNet121":

        feature_model = nn.Sequential(
            model.features,
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(
                (1, 1)
            ),
            nn.Flatten()
        )

    elif model_name == "EfficientNet-B0":

        feature_model = nn.Sequential(
            model.features,
            model.avgpool,
            nn.Flatten()
        )

    else:

        raise ValueError(
            "Unsupported model"
        )

    return feature_model.to(DEVICE)


feature_extractor = get_feature_extractor(
    best_model,
    best_model_name
)


# ============================================================
# 17. EXTRACT FEATURES
# ============================================================

def extract_features(
    loader,
    feature_extractor
):

    feature_extractor.eval()

    features = []
    labels_list = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            batch_features = (
                feature_extractor(images)
            )

            features.append(
                batch_features.cpu().numpy()
            )

            labels_list.append(
                labels.numpy()
            )

    features = np.concatenate(
        features,
        axis=0
    )

    labels = np.concatenate(
        labels_list,
        axis=0
    )

    return features, labels


print("\n")
print("=" * 70)
print("EXTRACTING DEEP FEATURES")
print("=" * 70)


train_features, train_labels = extract_features(
    train_loader,
    feature_extractor
)


test_features, test_labels = extract_features(
    test_loader,
    feature_extractor
)


print(
    "Training feature shape:",
    train_features.shape
)

print(
    "Test feature shape:",
    test_features.shape
)


# ============================================================
# 18. SCALE FEATURES
# ============================================================

scaler = StandardScaler()

train_features_scaled = scaler.fit_transform(
    train_features
)

test_features_scaled = scaler.transform(
    test_features
)


# ============================================================
# 19. TABLE 2
# DIFFERENT CLASSIFIERS
# ============================================================

classifiers = {

    "Logistic Regression":
        LogisticRegression(
            max_iter=1000,
            random_state=SEED
        ),

    "Decision Tree":
        DecisionTreeClassifier(
            random_state=SEED
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            random_state=SEED,
            n_jobs=-1
        ),

    "KNN":
        KNeighborsClassifier(
            n_neighbors=5,
            n_jobs=-1
        ),

    "Linear SVM":
        SVC(
            kernel="linear",
            probability=True,
            random_state=SEED
        ),

    "RBF-SVM":
        SVC(
            kernel="rbf",
            probability=True,
            random_state=SEED
        ),

    "XGBoost":
        XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",
            num_class=NUM_CLASSES,
            eval_metric="mlogloss",
            random_state=SEED,
            n_jobs=-1
        )
}


table2_results = []


print("\n")
print("=" * 70)
print("TABLE 2 - CLASSIFIER COMPARISON")
print("=" * 70)


for classifier_name, classifier in classifiers.items():

    print("\n" + "-" * 70)
    print("Training classifier:", classifier_name)
    print("-" * 70)

    classifier.fit(
        train_features_scaled,
        train_labels
    )

    predictions = classifier.predict(
        test_features_scaled
    )

    if hasattr(
        classifier,
        "predict_proba"
    ):

        probabilities = (
            classifier.predict_proba(
                test_features_scaled
            )
        )

    else:

        probabilities = None

    accuracy = accuracy_score(
        test_labels,
        predictions
    )

    precision = precision_score(
        test_labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        test_labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        test_labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    if probabilities is not None:

        try:

            auc = roc_auc_score(
                test_labels,
                probabilities,
                multi_class="ovr",
                average="weighted"
            )

        except:

            auc = np.nan

    else:

        auc = np.nan

    table2_results.append({

        "Feature Extractor":
            "Deep Features",

        "Classifier":
            classifier_name,

        "Accuracy (%)":
            accuracy * 100,

        "Precision (%)":
            precision * 100,

        "Recall (%)":
            recall * 100,

        "F1-Score (%)":
            f1 * 100,

        "AUC (%)":
            auc * 100
            if not np.isnan(auc)
            else np.nan
    })

    print(
        f"Accuracy : {accuracy * 100:.2f}%"
    )

    print(
        f"Precision: {precision * 100:.2f}%"
    )

    print(
        f"Recall   : {recall * 100:.2f}%"
    )

    print(
        f"F1 Score : {f1 * 100:.2f}%"
    )

    if not np.isnan(auc):

        print(
            f"AUC      : {auc * 100:.2f}%"
        )


table2_df = pd.DataFrame(
    table2_results
)


print("\n")
print("=" * 70)
print("TABLE 2 RESULTS")
print("=" * 70)

print(
    table2_df.to_string(
        index=False
    )
)


# ============================================================
# 20. TABLE 3
# COMPUTATIONAL EFFICIENCY
# ============================================================

TABLE3_MODELS = [
    "AlexNet",
    "VGG16",
    "VGG19",
    "ResNet18",
    "ResNet50",
    "DenseNet121",
    "EfficientNet-B0"
]


def count_parameters(model):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return total, trainable


def calculate_model_size(model):

    total_bytes = 0

    for parameter in model.parameters():

        total_bytes += (
            parameter.numel()
            * parameter.element_size()
        )

    for buffer in model.buffers():

        total_bytes += (
            buffer.numel()
            * buffer.element_size()
        )

    return total_bytes / (
        1024 * 1024
    )


def calculate_flops(model):

    model.eval()

    dummy_input = torch.randn(
        1,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE
    ).to(DEVICE)

    try:

        flops, params = profile(
            model,
            inputs=(dummy_input,),
            verbose=False
        )

        return flops / 1e9

    except Exception as e:

        print(
            "FLOPs calculation failed:",
            e
        )

        return np.nan


def calculate_inference_time(
    model,
    loader,
    number_of_batches=10
):

    model.eval()

    times = []

    with torch.no_grad():

        for batch_number, (
            images,
            _
        ) in enumerate(loader):

            if batch_number >= number_of_batches:
                break

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            if DEVICE.type == "cuda":

                torch.cuda.synchronize()

            start_time = time.perf_counter()

            _ = model(images)

            if DEVICE.type == "cuda":

                torch.cuda.synchronize()

            end_time = time.perf_counter()

            elapsed = (
                end_time - start_time
            )

            batch_size = images.size(0)

            per_image_ms = (
                elapsed
                / batch_size
                * 1000
            )

            times.append(
                per_image_ms
            )

    if len(times) == 0:

        return np.nan

    return np.mean(times)


table3_results = []


print("\n")
print("=" * 70)
print("TABLE 3 - COMPUTATIONAL EFFICIENCY")
print("=" * 70)


for model_name in TABLE3_MODELS:

    print("\n" + "-" * 70)
    print(
        "Analyzing:",
        model_name
    )
    print("-" * 70)

    model = trained_models[
        model_name
    ]

    total_parameters, trainable_parameters = (
        count_parameters(model)
    )

    parameters_millions = (
        total_parameters / 1e6
    )

    model_size_mb = calculate_model_size(
        model
    )

    flops_g = calculate_flops(
        model
    )

    inference_ms = calculate_inference_time(
        model,
        test_loader,
        number_of_batches=10
    )

    accuracy_row = table1_df[
        table1_df["Model"] == model_name
    ]

    if len(accuracy_row) > 0:

        accuracy_value = accuracy_row[
            "Accuracy (%)"
        ].iloc[0]

    else:

        accuracy_value = np.nan

    table3_results.append({

        "Model":
            model_name,

        "Parameters (M)":
            parameters_millions,

        "Model Size (MB)":
            model_size_mb,

        "FLOPs (G)":
            flops_g,

        "Inference Time (ms)":
            inference_ms,

        "Accuracy (%)":
            accuracy_value
    })

    print(
        f"Parameters : {parameters_millions:.2f} M"
    )

    print(
        f"Model Size : {model_size_mb:.2f} MB"
    )

    if not np.isnan(flops_g):

        print(
            f"FLOPs      : {flops_g:.2f} G"
        )

    else:

        print(
            "FLOPs      : N/A"
        )

    print(
        f"Inference  : {inference_ms:.3f} ms"
    )

    print(
        f"Accuracy   : {accuracy_value:.2f}%"
    )


table3_df = pd.DataFrame(
    table3_results
)


print("\n")
print("=" * 70)
print("TABLE 3 RESULTS")
print("=" * 70)

print(
    table3_df.to_string(
        index=False
    )
)


# ============================================================
# 21. CREATE RESULTS DIRECTORY
# ============================================================

RESULTS_DIR = "results"

MODELS_DIR = os.path.join(
    RESULTS_DIR,
    "models"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

os.makedirs(
    MODELS_DIR,
    exist_ok=True
)


# ============================================================
# 22. SAVE EXCEL FILE
# ============================================================

excel_path = os.path.join(
    RESULTS_DIR,
    "All_Tables.xlsx"
)


with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    table1_df.to_excel(
        writer,
        sheet_name="Table 1",
        index=False
    )

    table2_df.to_excel(
        writer,
        sheet_name="Table 2",
        index=False
    )

    table3_df.to_excel(
        writer,
        sheet_name="Table 3",
        index=False
    )


print("\n")
print("=" * 70)
print("EXCEL FILE SAVED")
print("=" * 70)

print(
    os.path.abspath(excel_path)
)


# ============================================================
# 23. SAVE TRAINED PYTORCH MODELS
# ============================================================

for model_name, model in trained_models.items():

    safe_name = (
        model_name
        .replace("-", "_")
    )

    model_path = os.path.join(
        MODELS_DIR,
        safe_name + ".pth"
    )

    torch.save(
        model.state_dict(),
        model_path
    )


print("\nTrained models saved in:")

print(
    os.path.abspath(MODELS_DIR)
)


# ============================================================
# 24. SAVE CLASSIFIER RESULTS
# ============================================================

classifier_results_path = os.path.join(
    RESULTS_DIR,
    "classifier_results.csv"
)

table2_df.to_csv(
    classifier_results_path,
    index=False
)


# ============================================================
# 25. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print("\nTABLE 1:")
print(
    table1_df.to_string(
        index=False
    )
)

print("\nTABLE 2:")
print(
    table2_df.to_string(
        index=False
    )
)

print("\nTABLE 3:")
print(
    table3_df.to_string(
        index=False
    )
)

print("\n")
print("=" * 70)

print(
    "Best Transfer Learning Model:",
    best_model_name
)

print(
    "Best Accuracy:",
    f"{best_model_accuracy:.2f}%"
)

print(
    "\nExcel file:",
    os.path.abspath(excel_path)
)

print(
    "\nAll work completed successfully."
)

print("=" * 70)

DEVICE: cuda

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.

Dataset path:
/kaggle/input/skin-cancer9-classesisic

TRAIN ROOT:
/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Train

TEST ROOT:
/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration/Test

Classes found:
['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']

Number of classes: 9
Training images: 2239
Test images: 118

Train samples: 1903
Validation samples: 336
Test samples: 118


TABLE 1 - TRANSFER LEARNING MODELS

----------------------------------------------------------------------
Training: AlexNet
----------------------------------------------------------------------
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/

100%|██████████| 233M/233M [00:01<00:00, 178MB/s]


Epoch [1/5] Train Acc: 48.24% Val Acc: 55.36%
Epoch [2/5] Train Acc: 61.48% Val Acc: 66.96%
Epoch [3/5] Train Acc: 69.00% Val Acc: 64.88%
Epoch [4/5] Train Acc: 70.63% Val Acc: 66.37%
Epoch [5/5] Train Acc: 75.46% Val Acc: 65.18%

AlexNet Results:
Accuracy (%): 48.31
Precision (%): 53.04
Recall (%): 48.31
F1-Score (%): 44.22
AUC (%): 87.14

----------------------------------------------------------------------
Training: VGG16
----------------------------------------------------------------------
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 177MB/s]


Epoch [1/5] Train Acc: 41.15% Val Acc: 53.87%
Epoch [2/5] Train Acc: 59.06% Val Acc: 63.10%
Epoch [3/5] Train Acc: 64.48% Val Acc: 61.90%
Epoch [4/5] Train Acc: 69.42% Val Acc: 63.69%
Epoch [5/5] Train Acc: 71.47% Val Acc: 67.56%

VGG16 Results:
Accuracy (%): 51.69
Precision (%): 63.78
Recall (%): 51.69
F1-Score (%): 47.37
AUC (%): 87.62

----------------------------------------------------------------------
Training: VGG19
----------------------------------------------------------------------
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...

Epoch [1/5] Train Acc: 39.52% Val Acc: 58.33%
Epoch [2/5] Train Acc: 65.79% Val Acc: 67.26%
Epoch [3/5] Train Acc: 76.25% Val Acc: 68.45%
Epoch [4/5] Train Acc: 82.50% Val Acc: 68.75%
Epoch [5/5] Train Acc: 85.50% Val Acc: 69.94%

ResNet101 Results:
Accuracy (%): 56.78
Precision (%): 60.27
Recall (%): 56.78
F1-Score (%): 54.41
AUC (%): 87.10

----------------------------------------------------------------------
Training: DenseNet121
----------------------------------------------------------------------
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 176MB/s]


Epoch [1/5] Train Acc: 49.40% Val Acc: 64.88%
Epoch [2/5] Train Acc: 70.47% Val Acc: 69.05%
Epoch [3/5] Train Acc: 78.09% Val Acc: 68.15%
Epoch [4/5] Train Acc: 82.50% Val Acc: 70.24%
Epoch [5/5] Train Acc: 85.50% Val Acc: 69.64%

DenseNet121 Results:
Accuracy (%): 57.63
Precision (%): 62.28
Recall (%): 57.63
F1-Score (%): 55.56
AUC (%): 88.69

----------------------------------------------------------------------
Training: EfficientNet-B0
----------------------------------------------------------------------
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 151MB/s]


Epoch [1/5] Train Acc: 40.04% Val Acc: 53.57%
Epoch [2/5] Train Acc: 61.27% Val Acc: 65.77%
Epoch [3/5] Train Acc: 69.52% Val Acc: 69.05%
Epoch [4/5] Train Acc: 75.35% Val Acc: 70.83%
Epoch [5/5] Train Acc: 78.88% Val Acc: 69.94%

EfficientNet-B0 Results:
Accuracy (%): 60.17
Precision (%): 63.87
Recall (%): 60.17
F1-Score (%): 56.99
AUC (%): 90.42


TABLE 1 RESULTS
          Model  Accuracy (%)  Precision (%)  Recall (%)  F1-Score (%)   AUC (%)
        AlexNet     48.305085      53.041045   48.305085     44.223493 87.141402
          VGG16     51.694915      63.776938   51.694915     47.366096 87.621411
          VGG19     50.000000      59.720837   50.000000     45.322933 83.401679
       ResNet18     55.932203      56.268707   55.932203     51.894714 87.969714
       ResNet50     55.084746      56.410360   55.084746     52.556535 87.085121
      ResNet101     56.779661      60.270689   56.779661     54.407321 87.103761
    DenseNet121     57.627119      62.280431   57.627119     55.5